In [1]:
# -*- coding: utf-8 -*-
"""
Convert ER Mapper .ers rasters to Cloud Optimized GeoTIFFs (COG).

Input:
  .ers files

Output:
  *_cog.tif files
"""

import os
import time
import arcpy


# ============================================================
# USER SETTINGS
# ============================================================

INPUT_FOLDER = r"C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2025\kalk_prosjekt3.0\Geophysics\Mag_Grav_250m"

OUTPUT_FOLDER = r"D:\Kalk_project_3.0\Geophysics\Geophysics_ers_as_cog"

OVERWRITE_EXISTING = False


# ============================================================
# HELPERS
# ============================================================

def ts():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def msg(s):
    text = f"[{ts()}] {s}"
    try:
        arcpy.AddMessage(text)
    except Exception:
        pass
    print(text, flush=True)


def safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)
        elif os.path.exists(path):
            os.remove(path)
    except Exception:
        pass


def list_ers_files(folder):
    files = []

    for fn in os.listdir(folder):
        full = os.path.join(folder, fn)

        if os.path.isfile(full) and fn.lower().endswith(".ers"):
            files.append(full)

    return sorted(files)


def get_base_name(path):
    return os.path.splitext(os.path.basename(path))[0]


# ============================================================
# CORE
# ============================================================

def convert_one_ers_to_cog(in_ers, output_folder):
    base = get_base_name(in_ers)
    out_cog = os.path.join(output_folder, f"{base}_cog.tif")

    if os.path.exists(out_cog) and not OVERWRITE_EXISTING:
        msg(f"Skipping existing COG: {os.path.basename(out_cog)}")
        return

    msg("------------------------------------------------------------")
    msg(f"START: {os.path.basename(in_ers)}")
    msg(f"Output: {out_cog}")

    try:
        if OVERWRITE_EXISTING:
            safe_delete(out_cog)

        arcpy.management.CopyRaster(
            in_raster=in_ers,
            out_rasterdataset=out_cog,
            format="COG"
        )

        msg(f"END: {os.path.basename(in_ers)}")

    except Exception as e:
        msg(f"FAILED: {os.path.basename(in_ers)}")
        msg(str(e))
        raise

    finally:
        msg("------------------------------------------------------------")


# ============================================================
# MAIN
# ============================================================

def main():
    t0 = time.time()

    arcpy.env.overwriteOutput = OVERWRITE_EXISTING

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    ers_files = list_ers_files(INPUT_FOLDER)

    if len(ers_files) == 0:
        msg("No .ers files found.")
        msg(f"Checked folder: {INPUT_FOLDER}")
        return

    msg(f"Found {len(ers_files)} .ers files.")
    msg(f"Input folder: {INPUT_FOLDER}")
    msg(f"Output folder: {OUTPUT_FOLDER}")
    msg(f"Overwrite existing: {OVERWRITE_EXISTING}")

    for ers in ers_files:
        convert_one_ers_to_cog(ers, OUTPUT_FOLDER)

    msg("Finished converting all .ers files to COG.")
    msg(f"Total time: {(time.time() - t0) / 60:.1f} minutes")


if __name__ == "__main__":
    main()

[2026-05-13 21:22:01] Found 8 .ers files.
[2026-05-13 21:22:01] Input folder: C:\Users\acosta_pedro\OneDrive - Norges geologiske undersøkelse\Geochemistry NGU_2025\kalk_prosjekt3.0\Geophysics\Mag_Grav_250m
[2026-05-13 21:22:01] Output folder: D:\Kalk_project_3.0\Geophysics\Geophysics_ers_as_cog
[2026-05-13 21:22:01] Overwrite existing: False
[2026-05-13 21:22:01] ------------------------------------------------------------
[2026-05-13 21:22:01] START: GRAV_BOUGUER_ANOMALY_NORWAY_500m.ers
[2026-05-13 21:22:01] Output: D:\Kalk_project_3.0\Geophysics\Geophysics_ers_as_cog\GRAV_BOUGUER_ANOMALY_NORWAY_500m_cog.tif
[2026-05-13 21:22:03] END: GRAV_BOUGUER_ANOMALY_NORWAY_500m.ers
[2026-05-13 21:22:03] ------------------------------------------------------------
[2026-05-13 21:22:03] ------------------------------------------------------------
[2026-05-13 21:22:03] START: HG_GRAV_BOUGUER_ANOMALY_NORWAY_500m.ers
[2026-05-13 21:22:03] Output: D:\Kalk_project_3.0\Geophysics\Geophysics_ers_as_cog\H

In [3]:
# -*- coding: utf-8 -*-
"""
Resample and mask existing GeoTIFF / COG rasters to match a MASK raster.

INPUT:
    Existing GeoTIFF / COG rasters

WORKFLOW:
    1. Read GeoTIFF / COG rasters
    2. Project / resample to MASK grid
    3. Apply MASK
    4. Save final masked COG

FEATURES:
    - Robust skip logic
    - Recognizes previous output naming styles
    - Safe temp cleanup
    - Exact grid matching to MASK raster

FINAL OUTPUTS:
    *_10m_masked_cog.tif
"""

import os
import time
import hashlib
import arcpy

#from arcpy.sa import ExtractByMask
from arcpy.sa import SetNull


# ============================================================
# USER SETTINGS
# ============================================================


INPUT_FOLDER = r"G:\Geology_rasters_2026\Outcrop_map"

MASK_RASTER = r"E:\mask\Mask_land_Kalk_cog.tif"

OUTPUT_FOLDER = r"G:\Geology_rasters_2026\Outcrop_map\Resampled"

# ------------------------------------------------------------
# TOGGLES
# ------------------------------------------------------------

# Skip if ANY previous matching output already exists
SKIP_IF_OUTPUT_EXISTS = True

# Overwrite outputs if they already exist
OVERWRITE_OUTPUTS = False

# Keep TIFF if COG creation fails
KEEP_TIFF_IF_COG_FAILS = True

# Resampling method
# Continuous: BILINEAR or CUBIC
# Categorical: NEAREST
RESAMPLING_METHOD = "BILINEAR"

VALID_EXTENSIONS = (".tif", ".tiff")


# ============================================================
# TEMP WORKSPACE
# ============================================================

TEMP_ROOT = r"C:\arc_tmp_resample"

TEMP_GDB = os.path.join(TEMP_ROOT, "temp.gdb")

TEMP_STAGE = os.path.join(TEMP_ROOT, "stage")


# ============================================================
# HELPERS
# ============================================================

def ts():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def msg(s):
    text = f"[{ts()}] {s}"

    try:
        arcpy.AddMessage(text)
    except Exception:
        pass

    print(text, flush=True)


def safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)

        elif os.path.exists(path):
            os.remove(path)

    except Exception:
        pass


def ensure_folders():

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(TEMP_ROOT, exist_ok=True)
    os.makedirs(TEMP_STAGE, exist_ok=True)

    if not arcpy.Exists(TEMP_GDB):
        arcpy.management.CreateFileGDB(TEMP_ROOT, "temp.gdb")


def list_input_rasters(folder):

    rasters = []

    for fn in os.listdir(folder):

        full = os.path.join(folder, fn)

        if not os.path.isfile(full):
            continue

        if fn.lower().endswith(VALID_EXTENSIONS):
            rasters.append(full)

    return sorted(rasters)


def get_base_name(path):
    return os.path.splitext(os.path.basename(path))[0]


def clean_base_name(base):

    suffixes = [
        "_source_cog",
        "_cog"
    ]

    for suffix in suffixes:

        if base.lower().endswith(suffix):
            base = base[:-len(suffix)]

    return base


def short_safe_id(name):

    h = hashlib.md5(name.encode("utf-8")).hexdigest()[:8]

    return f"r_{h}"


def report_raster_info(label, raster_path):

    if not arcpy.Exists(raster_path):
        msg(f"{label}: not found")
        return

    r = arcpy.Raster(raster_path)

    e = r.extent

    msg(f"{label}:")
    msg(f"  rows/cols   = {r.height} x {r.width}")
    msg(f"  cell size   = {r.meanCellWidth} x {r.meanCellHeight}")
    msg(f"  extent      = XMin={e.XMin}, YMin={e.YMin}, XMax={e.XMax}, YMax={e.YMax}")


def compare_to_mask(mask_path, out_path, tol=1e-9):

    if not arcpy.Exists(mask_path):
        return False

    if not arcpy.Exists(out_path):
        return False

    m = arcpy.Raster(mask_path)

    r = arcpy.Raster(out_path)

    me = m.extent
    re = r.extent

    same_rows = (m.height == r.height)
    same_cols = (m.width == r.width)

    same_cx = abs(m.meanCellWidth - r.meanCellWidth) < tol
    same_cy = abs(m.meanCellHeight - r.meanCellHeight) < tol

    same_xmin = abs(me.XMin - re.XMin) < tol
    same_ymin = abs(me.YMin - re.YMin) < tol
    same_xmax = abs(me.XMax - re.XMax) < tol
    same_ymax = abs(me.YMax - re.YMax) < tol

    ok = all([
        same_rows,
        same_cols,
        same_cx,
        same_cy,
        same_xmin,
        same_ymin,
        same_xmax,
        same_ymax
    ])

    if ok:
        msg(f"{os.path.basename(out_path)} matches MASK exactly.")

    else:
        msg(f"{os.path.basename(out_path)} does NOT match MASK exactly.")

    return ok


# ============================================================
# SKIP LOGIC
# ============================================================

def find_existing_output(output_folder, base, original_base):

    possible_names = [

        # Current naming style
        f"{base}_10m_masked_cog.tif",

        # Previous naming styles
        f"{original_base}_10m_cog.tif",
        f"{base}_10m_cog.tif",

        # TIFF fallbacks
        f"{base}_10m_masked.tif",
        f"{base}_10m.tif",
        f"{original_base}_10m.tif"
    ]

    for name in possible_names:

        path = os.path.join(output_folder, name)

        if os.path.exists(path):
            return path

    return None


# ============================================================
# CORE PROCESSING
# ============================================================

def process_one_raster(
    in_raster,
    mask_raster,
    output_folder,
    mask_sr,
    mask_cellsize,
    resampling_method,
    scratch_gdb
):

    original_base = get_base_name(in_raster)

    base = clean_base_name(original_base)

    rid = short_safe_id(base)

    final_cog = os.path.join(
        output_folder,
        f"{base}_10m_masked_cog.tif"
    )

    fallback_tif = os.path.join(
        output_folder,
        f"{base}_10m_masked.tif"
    )

    # --------------------------------------------------------
    # SKIP LOGIC
    # --------------------------------------------------------

    existing_output = find_existing_output(
        output_folder=output_folder,
        base=base,
        original_base=original_base
    )

    if SKIP_IF_OUTPUT_EXISTS and existing_output is not None:

        msg(
            f"Skipping existing output: "
            f"{os.path.basename(existing_output)}"
        )

        return

    # --------------------------------------------------------
    # OVERWRITE
    # --------------------------------------------------------

    if OVERWRITE_OUTPUTS:

        safe_delete(final_cog)

        safe_delete(fallback_tif)

    # --------------------------------------------------------
    # TEMP FILES
    # --------------------------------------------------------

    tmp_proj = os.path.join(
        scratch_gdb,
        f"{rid}_p"
    )

    tmp_mask = os.path.join(
        scratch_gdb,
        f"{rid}_m"
    )

    stage_tif = os.path.join(
        TEMP_STAGE,
        f"{rid}_stage.tif"
    )

    safe_delete(tmp_proj)
    safe_delete(tmp_mask)
    safe_delete(stage_tif)

    msg("------------------------------------------------------------")
    msg(f"START: {os.path.basename(in_raster)}")
    msg(f"Final output: {final_cog}")

    try:

        # ----------------------------------------------------
        # PROJECT / RESAMPLE
        # ----------------------------------------------------

        msg("Projecting / resampling to MASK grid...")

        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):

            arcpy.management.ProjectRaster(
                in_raster=in_raster,
                out_raster=tmp_proj,
                out_coor_system=mask_sr,
                resampling_type=resampling_method,
                cell_size=mask_cellsize
            )

       # ----------------------------------------------------
        # MASK
        # ----------------------------------------------------

        msg("Applying MASK: outside Norway -> NoData...")

        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):

            mask = arcpy.Raster(mask_raster)
            projected = arcpy.Raster(tmp_proj)

            # Mask == 1 : keep original raster value
            # Mask != 1 : set to NoData
            masked = SetNull(
                mask != 1,
                projected
            )

            masked.save(tmp_mask)

        # ----------------------------------------------------
        # STAGE TIFF
        # ----------------------------------------------------

        msg(f"Writing staged TIFF: {stage_tif}")

        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):

            arcpy.management.CopyRaster(
                in_raster=tmp_mask,
                out_rasterdataset=stage_tif,
                format="TIFF"
            )

        # ----------------------------------------------------
        # FINAL COG
        # ----------------------------------------------------

        msg(f"Creating final masked COG: {final_cog}")

        try:

            with arcpy.EnvManager(
                snapRaster=mask_raster,
                cellSize=mask_raster,
                extent=mask_raster,
                outputCoordinateSystem=mask_sr
            ):

                arcpy.management.CopyRaster(
                    in_raster=stage_tif,
                    out_rasterdataset=final_cog,
                    format="COG"
                )

            compare_to_mask(
                mask_raster,
                final_cog
            )

            safe_delete(stage_tif)

        except Exception as cog_err:

            msg("COG creation failed.")
            msg(str(cog_err))

            if KEEP_TIFF_IF_COG_FAILS:

                msg(
                    f"Keeping regular TIFF instead: "
                    f"{fallback_tif}"
                )

                safe_delete(fallback_tif)

                arcpy.management.CopyRaster(
                    in_raster=stage_tif,
                    out_rasterdataset=fallback_tif,
                    format="TIFF"
                )

                compare_to_mask(
                    mask_raster,
                    fallback_tif
                )

            else:
                raise

        msg(f"END: {os.path.basename(in_raster)}")

    except Exception as e:

        msg(f"FAILED: {os.path.basename(in_raster)}")

        msg(str(e))

        raise

    finally:

        safe_delete(tmp_proj)
        safe_delete(tmp_mask)

        msg("------------------------------------------------------------")


# ============================================================
# MAIN
# ============================================================

def main():

    t0 = time.time()

    arcpy.env.overwriteOutput = True

    arcpy.CheckOutExtension("Spatial")

    ensure_folders()

    mask_desc = arcpy.Describe(MASK_RASTER)

    mask_sr = mask_desc.spatialReference

    mask_ras = arcpy.Raster(MASK_RASTER)

    mask_cellsize = mask_ras.meanCellWidth

    if abs(
        mask_ras.meanCellWidth -
        mask_ras.meanCellHeight
    ) > 1e-9:

        raise ValueError(
            "MASK raster does not have square cells."
        )

    arcpy.env.snapRaster = MASK_RASTER
    arcpy.env.cellSize = MASK_RASTER
    arcpy.env.extent = MASK_RASTER
    arcpy.env.outputCoordinateSystem = mask_sr

    msg("MASK reference:")

    report_raster_info(
        "MASK",
        MASK_RASTER
    )

    rasters = list_input_rasters(INPUT_FOLDER)

    if len(rasters) == 0:

        msg("No GeoTIFF/COG rasters found.")

        msg(f"Checked folder: {INPUT_FOLDER}")

        return

    msg(f"Found {len(rasters)} GeoTIFF/COG rasters to process.")
    msg(f"Input folder: {INPUT_FOLDER}")
    msg(f"Output folder: {OUTPUT_FOLDER}")
    msg(f"Resampling method: {RESAMPLING_METHOD}")
    msg(f"Target cell size: {mask_cellsize}")
    msg(f"Skip existing outputs: {SKIP_IF_OUTPUT_EXISTS}")
    msg(f"Overwrite outputs: {OVERWRITE_OUTPUTS}")
    msg(f"Keep TIFF if COG fails: {KEEP_TIFF_IF_COG_FAILS}")

    scratch_gdb = TEMP_GDB

    for r in rasters:

        process_one_raster(
            in_raster=r,
            mask_raster=MASK_RASTER,
            output_folder=OUTPUT_FOLDER,
            mask_sr=mask_sr,
            mask_cellsize=mask_cellsize,
            resampling_method=RESAMPLING_METHOD,
            scratch_gdb=scratch_gdb
        )

    msg("Finished all rasters.")

    msg(
        f"Total time: "
        f"{(time.time() - t0) / 60:.1f} minutes"
    )


if __name__ == "__main__":
    main()

[2026-09-07 20:45:36] MASK reference:
[2026-09-07 20:45:36] MASK:
[2026-09-07 20:45:36]   rows/cols   = 162232 x 135236
[2026-09-07 20:45:36]   cell size   = 10.0 x 10.0
[2026-09-07 20:45:36]   extent      = XMin=-154809.95275266352, YMin=6398344.6625600215, XMax=1197550.0472473365, YMax=8020664.6625600215
[2026-09-07 20:45:36] Found 1 GeoTIFF/COG rasters to process.
[2026-09-07 20:45:36] Input folder: G:\Geology_rasters_2026\Outcrop_map
[2026-09-07 20:45:36] Output folder: G:\Geology_rasters_2026\Outcrop_map\Resampled
[2026-09-07 20:45:36] Resampling method: BILINEAR
[2026-09-07 20:45:36] Target cell size: 10.0
[2026-09-07 20:45:36] Skip existing outputs: True
[2026-09-07 20:45:36] Overwrite outputs: False
[2026-09-07 20:45:36] Keep TIFF if COG fails: True
[2026-09-07 20:45:36] ------------------------------------------------------------
[2026-09-07 20:45:36] START: Losm_fjell_prediksjonskart_10x10m_8bit_2026_scores.tif
[2026-09-07 20:45:36] Final output: G:\Geology_rasters_2026\Outcr

In [2]:
# -*- coding: utf-8 -*-
"""
Resample and mask existing Water forekomst (water Ca content) GeoTIFF / COG rasters to match a MASK raster.

INPUT:
    Existing GeoTIFF / COG rasters

WORKFLOW:
    1. Read GeoTIFF / COG rasters
    2. Project / resample to MASK grid
    3. Apply MASK
    4. Save final masked COG

FEATURES:
    - Robust skip logic
    - Recognizes previous output naming styles
    - Safe temp cleanup
    - Exact grid matching to MASK raster

FINAL OUTPUTS:
    *_10m_masked_cog.tif
"""

import os
import time
import hashlib
import arcpy

from arcpy.sa import ExtractByMask


# ============================================================
# USER SETTINGS
# ============================================================


INPUT_FOLDER = r"N:\Prosjekter\411200_Kalk_i_grunnen\ElvInnsjo_Kalsium\to_resample"

MASK_RASTER = r"E:\mask\Mask_land_Kalk_cog.tif"

OUTPUT_FOLDER = r"G:\Geology_rasters_2026\WaterCa"

# ------------------------------------------------------------
# TOGGLES
# ------------------------------------------------------------

# Skip if ANY previous matching output already exists
SKIP_IF_OUTPUT_EXISTS = True

# Overwrite outputs if they already exist
OVERWRITE_OUTPUTS = False

# Keep TIFF if COG creation fails
KEEP_TIFF_IF_COG_FAILS = True

# Resampling method
# Continuous: BILINEAR or CUBIC
# Categorical: NEAREST
RESAMPLING_METHOD = "BILINEAR"

VALID_EXTENSIONS = (".tif", ".tiff")


# ============================================================
# TEMP WORKSPACE
# ============================================================

TEMP_ROOT = r"C:\arc_tmp_resample"

TEMP_GDB = os.path.join(TEMP_ROOT, "temp.gdb")

TEMP_STAGE = os.path.join(TEMP_ROOT, "stage")


# ============================================================
# HELPERS
# ============================================================

def ts():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def msg(s):
    text = f"[{ts()}] {s}"

    try:
        arcpy.AddMessage(text)
    except Exception:
        pass

    print(text, flush=True)


def safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)

        elif os.path.exists(path):
            os.remove(path)

    except Exception:
        pass


def ensure_folders():

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(TEMP_ROOT, exist_ok=True)
    os.makedirs(TEMP_STAGE, exist_ok=True)

    if not arcpy.Exists(TEMP_GDB):
        arcpy.management.CreateFileGDB(TEMP_ROOT, "temp.gdb")


def list_input_rasters(folder):

    rasters = []

    for fn in os.listdir(folder):

        full = os.path.join(folder, fn)

        if not os.path.isfile(full):
            continue

        if fn.lower().endswith(VALID_EXTENSIONS):
            rasters.append(full)

    return sorted(rasters)


def get_base_name(path):
    return os.path.splitext(os.path.basename(path))[0]


def clean_base_name(base):

    suffixes = [
        "_source_cog",
        "_cog"
    ]

    for suffix in suffixes:

        if base.lower().endswith(suffix):
            base = base[:-len(suffix)]

    return base


def short_safe_id(name):

    h = hashlib.md5(name.encode("utf-8")).hexdigest()[:8]

    return f"r_{h}"


def report_raster_info(label, raster_path):

    if not arcpy.Exists(raster_path):
        msg(f"{label}: not found")
        return

    r = arcpy.Raster(raster_path)

    e = r.extent

    msg(f"{label}:")
    msg(f"  rows/cols   = {r.height} x {r.width}")
    msg(f"  cell size   = {r.meanCellWidth} x {r.meanCellHeight}")
    msg(f"  extent      = XMin={e.XMin}, YMin={e.YMin}, XMax={e.XMax}, YMax={e.YMax}")


def compare_to_mask(mask_path, out_path, tol=1e-9):

    if not arcpy.Exists(mask_path):
        return False

    if not arcpy.Exists(out_path):
        return False

    m = arcpy.Raster(mask_path)

    r = arcpy.Raster(out_path)

    me = m.extent
    re = r.extent

    same_rows = (m.height == r.height)
    same_cols = (m.width == r.width)

    same_cx = abs(m.meanCellWidth - r.meanCellWidth) < tol
    same_cy = abs(m.meanCellHeight - r.meanCellHeight) < tol

    same_xmin = abs(me.XMin - re.XMin) < tol
    same_ymin = abs(me.YMin - re.YMin) < tol
    same_xmax = abs(me.XMax - re.XMax) < tol
    same_ymax = abs(me.YMax - re.YMax) < tol

    ok = all([
        same_rows,
        same_cols,
        same_cx,
        same_cy,
        same_xmin,
        same_ymin,
        same_xmax,
        same_ymax
    ])

    if ok:
        msg(f"{os.path.basename(out_path)} matches MASK exactly.")

    else:
        msg(f"{os.path.basename(out_path)} does NOT match MASK exactly.")

    return ok


# ============================================================
# SKIP LOGIC
# ============================================================

def find_existing_output(output_folder, base, original_base):

    possible_names = [

        # Current naming style
        f"{base}_10m_masked_cog.tif",

        # Previous naming styles
        f"{original_base}_10m_cog.tif",
        f"{base}_10m_cog.tif",

        # TIFF fallbacks
        f"{base}_10m_masked.tif",
        f"{base}_10m.tif",
        f"{original_base}_10m.tif"
    ]

    for name in possible_names:

        path = os.path.join(output_folder, name)

        if os.path.exists(path):
            return path

    return None


# ============================================================
# CORE PROCESSING
# ============================================================

def process_one_raster(
    in_raster,
    mask_raster,
    output_folder,
    mask_sr,
    mask_cellsize,
    resampling_method,
    scratch_gdb
):

    original_base = get_base_name(in_raster)

    base = clean_base_name(original_base)

    rid = short_safe_id(base)

    final_cog = os.path.join(
        output_folder,
        f"{base}_10m_masked_cog.tif"
    )

    fallback_tif = os.path.join(
        output_folder,
        f"{base}_10m_masked.tif"
    )

    # --------------------------------------------------------
    # SKIP LOGIC
    # --------------------------------------------------------

    existing_output = find_existing_output(
        output_folder=output_folder,
        base=base,
        original_base=original_base
    )

    if SKIP_IF_OUTPUT_EXISTS and existing_output is not None:

        msg(
            f"Skipping existing output: "
            f"{os.path.basename(existing_output)}"
        )

        return

    # --------------------------------------------------------
    # OVERWRITE
    # --------------------------------------------------------

    if OVERWRITE_OUTPUTS:

        safe_delete(final_cog)

        safe_delete(fallback_tif)

    # --------------------------------------------------------
    # TEMP FILES
    # --------------------------------------------------------

    tmp_proj = os.path.join(
        scratch_gdb,
        f"{rid}_p"
    )

    tmp_mask = os.path.join(
        scratch_gdb,
        f"{rid}_m"
    )

    stage_tif = os.path.join(
        TEMP_STAGE,
        f"{rid}_stage.tif"
    )

    safe_delete(tmp_proj)
    safe_delete(tmp_mask)
    safe_delete(stage_tif)

    msg("------------------------------------------------------------")
    msg(f"START: {os.path.basename(in_raster)}")
    msg(f"Final output: {final_cog}")

    try:

        # ----------------------------------------------------
        # PROJECT / RESAMPLE
        # ----------------------------------------------------

        msg("Projecting / resampling to MASK grid...")

        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):

            arcpy.management.ProjectRaster(
                in_raster=in_raster,
                out_raster=tmp_proj,
                out_coor_system=mask_sr,
                resampling_type=resampling_method,
                cell_size=mask_cellsize
            )

        # ----------------------------------------------------
        # MASK
        # ----------------------------------------------------

        msg("Applying MASK...")

        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):

            ExtractByMask(
                tmp_proj,
                mask_raster
            ).save(tmp_mask)

        # ----------------------------------------------------
        # STAGE TIFF
        # ----------------------------------------------------

        msg(f"Writing staged TIFF: {stage_tif}")

        with arcpy.EnvManager(
            snapRaster=mask_raster,
            cellSize=mask_raster,
            extent=mask_raster,
            outputCoordinateSystem=mask_sr
        ):

            arcpy.management.CopyRaster(
                in_raster=tmp_mask,
                out_rasterdataset=stage_tif,
                format="TIFF"
            )

        # ----------------------------------------------------
        # FINAL COG
        # ----------------------------------------------------

        msg(f"Creating final masked COG: {final_cog}")

        try:

            with arcpy.EnvManager(
                snapRaster=mask_raster,
                cellSize=mask_raster,
                extent=mask_raster,
                outputCoordinateSystem=mask_sr
            ):

                arcpy.management.CopyRaster(
                    in_raster=stage_tif,
                    out_rasterdataset=final_cog,
                    format="COG"
                )

            compare_to_mask(
                mask_raster,
                final_cog
            )

            safe_delete(stage_tif)

        except Exception as cog_err:

            msg("COG creation failed.")
            msg(str(cog_err))

            if KEEP_TIFF_IF_COG_FAILS:

                msg(
                    f"Keeping regular TIFF instead: "
                    f"{fallback_tif}"
                )

                safe_delete(fallback_tif)

                arcpy.management.CopyRaster(
                    in_raster=stage_tif,
                    out_rasterdataset=fallback_tif,
                    format="TIFF"
                )

                compare_to_mask(
                    mask_raster,
                    fallback_tif
                )

            else:
                raise

        msg(f"END: {os.path.basename(in_raster)}")

    except Exception as e:

        msg(f"FAILED: {os.path.basename(in_raster)}")

        msg(str(e))

        raise

    finally:

        safe_delete(tmp_proj)
        safe_delete(tmp_mask)

        msg("------------------------------------------------------------")


# ============================================================
# MAIN
# ============================================================

def main():

    t0 = time.time()

    arcpy.env.overwriteOutput = True

    arcpy.CheckOutExtension("Spatial")

    ensure_folders()

    mask_desc = arcpy.Describe(MASK_RASTER)

    mask_sr = mask_desc.spatialReference

    mask_ras = arcpy.Raster(MASK_RASTER)

    mask_cellsize = mask_ras.meanCellWidth

    if abs(
        mask_ras.meanCellWidth -
        mask_ras.meanCellHeight
    ) > 1e-9:

        raise ValueError(
            "MASK raster does not have square cells."
        )

    arcpy.env.snapRaster = MASK_RASTER
    arcpy.env.cellSize = MASK_RASTER
    arcpy.env.extent = MASK_RASTER
    arcpy.env.outputCoordinateSystem = mask_sr

    msg("MASK reference:")

    report_raster_info(
        "MASK",
        MASK_RASTER
    )

    rasters = list_input_rasters(INPUT_FOLDER)

    if len(rasters) == 0:

        msg("No GeoTIFF/COG rasters found.")

        msg(f"Checked folder: {INPUT_FOLDER}")

        return

    msg(f"Found {len(rasters)} GeoTIFF/COG rasters to process.")
    msg(f"Input folder: {INPUT_FOLDER}")
    msg(f"Output folder: {OUTPUT_FOLDER}")
    msg(f"Resampling method: {RESAMPLING_METHOD}")
    msg(f"Target cell size: {mask_cellsize}")
    msg(f"Skip existing outputs: {SKIP_IF_OUTPUT_EXISTS}")
    msg(f"Overwrite outputs: {OVERWRITE_OUTPUTS}")
    msg(f"Keep TIFF if COG fails: {KEEP_TIFF_IF_COG_FAILS}")

    scratch_gdb = TEMP_GDB

    for r in rasters:

        process_one_raster(
            in_raster=r,
            mask_raster=MASK_RASTER,
            output_folder=OUTPUT_FOLDER,
            mask_sr=mask_sr,
            mask_cellsize=mask_cellsize,
            resampling_method=RESAMPLING_METHOD,
            scratch_gdb=scratch_gdb
        )

    msg("Finished all rasters.")

    msg(
        f"Total time: "
        f"{(time.time() - t0) / 60:.1f} minutes"
    )


if __name__ == "__main__":
    main()

[2026-06-09 09:19:34] MASK reference:
[2026-06-09 09:19:34] MASK:
[2026-06-09 09:19:34]   rows/cols   = 162232 x 135236
[2026-06-09 09:19:34]   cell size   = 10.0 x 10.0
[2026-06-09 09:19:34]   extent      = XMin=-154809.95275266352, YMin=6398344.6625600215, XMax=1197550.0472473365, YMax=8020664.6625600215
[2026-06-09 09:19:34] Found 1 GeoTIFF/COG rasters to process.
[2026-06-09 09:19:34] Input folder: N:\Prosjekter\411200_Kalk_i_grunnen\ElvInnsjo_Kalsium\to_resample
[2026-06-09 09:19:34] Output folder: G:\Geology_rasters_2026\WaterCa
[2026-06-09 09:19:34] Resampling method: BILINEAR
[2026-06-09 09:19:34] Target cell size: 10.0
[2026-06-09 09:19:34] Skip existing outputs: True
[2026-06-09 09:19:34] Overwrite outputs: False
[2026-06-09 09:19:34] Keep TIFF if COG fails: True
[2026-06-09 09:19:34] ------------------------------------------------------------
[2026-06-09 09:19:34] START: KalsiumElvInnsjo4.tif
[2026-06-09 09:19:34] Final output: G:\Geology_rasters_2026\WaterCa\KalsiumElvInns